# R1-M1 — shared trainer: Colab verification

Run all cells top to bottom (**Runtime → Run all**, with a **GPU runtime** such as the T4). Nothing here needs the dataset.

What it does:
1. mounts Drive and unpacks `src/`, `scripts/`, `tests/` from `echoproof_sync.zip` (never touches `data/` or `reports/`);
2. checks the runtime and imports the trainer;
3. runs the 16 trainer unit tests on this runtime;
4. exercises what CPU tests cannot: fp16 mixed precision, DataLoader workers, a real two-layer BiGRU under deterministic
   and non-deterministic CUDA settings, and exact resume on GPU;
5. writes one evidence file to `reports/r1/r1_m1_colab_<timestamp>.json` and prints a verdict.

Failures in a case are recorded, not raised, so one run shows every problem at once.

In [ ]:
import os, sys, json, subprocess, zipfile, time, hashlib, tempfile, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

TRAINER = ROOT / "src" / "echoproof" / "training" / "trainer.py"
assert TRAINER.exists(), "src/echoproof/training/trainer.py missing - upload echoproof_sync.zip to the project folder"
print("project root:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project root: /content/drive/MyDrive/PAI&AIES Project


In [ ]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, pyarrow, torch
ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": numpy.__version__,
       "pyarrow": pyarrow.__version__, "cuda_available": torch.cuda.is_available(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if torch.cuda.is_available() else None}
print(json.dumps(ENV, indent=2))
if not ENV["cuda_available"]:
    print("\nWARNING: no GPU. GPU-only cases (fp16) will be skipped; switch Runtime > Change runtime type > T4 GPU for the full check.")

import echoproof.training.trainer as trainer_module
print("trainer module:", trainer_module.__file__)
TRAINER_SHA256 = hashlib.sha256(TRAINER.read_bytes()).hexdigest()

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "numpy": "2.4.6",
  "pyarrow": "23.0.1",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "vram_gb": 14.6
}
trainer module: /content/drive/MyDrive/PAI&AIES Project/src/echoproof/training/trainer.py


## 1. Unit tests on this runtime

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_m1_trainer.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-2500:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
summary_line = [l for l in proc.stdout.splitlines() if " passed" in l or " failed" in l]
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", summary_line[-1:] )

...................                                                      [100%]
19 passed in 14.20s

unit tests: PASSED ['19 passed in 14.20s']


## 2. Paths the CPU tests cannot cover

Synthetic 1.5 s clips (tone vs. harmonic-rich signal), 4 epochs. Two small models:
`TinyDetector` (conv + masked mean) and `SpectralBiGRU` (log power spectrogram -> the repo's real two-layer `BiGRU`,
packed sequences). The BiGRU here is a trainer smoke test, not the final feature pipeline.

In [ ]:
import numpy as np
from torch import nn
from echoproof.contracts import Chunk
from echoproof.models.bigru import BiGRU
from echoproof.training.datasets import ChunkDataset, CropDataset, DictWaveformLoader
from echoproof.training.trainer import TrainConfig, Trainer

RATE, CROP = 16000, 16000
WORK = Path(tempfile.mkdtemp(prefix="r1_m1_"))      # local disk: fast, discarded with the runtime


def make_data(n=32, seconds=1.5, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(int(RATE * seconds)) / RATE
    waves, labels = {}, {}
    for i in range(n):
        label = i % 2
        f0 = 220 if label == 0 else 2200
        x = np.sin(2 * np.pi * f0 * t + rng.uniform(0, 6)) + (0.5 * np.sin(2 * np.pi * 3 * f0 * t) if label else 0)
        waves[f"rec:{i:03d}"] = (0.3 * x + 0.02 * rng.standard_normal(len(t))).astype(np.float32)
        labels[f"rec:{i:03d}"] = label
    loader = DictWaveformLoader(waves)
    train = CropDataset(list(labels.items()), loader, seed=42, crop_samples=CROP)
    chunks = [Chunk(1, f"chunk:B:{rid}:{s}", rid, "speaker:1", "protocol:B", "validation", s, s + CROP, RATE, s / RATE,
                    (s + CROP) / RATE, CROP, CROP, "a" * 64) for rid in sorted(labels) for s in (0, len(waves[rid]) - CROP)]
    return train, ChunkDataset(chunks, labels, loader)


class TinyDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv, self.dropout, self.head = nn.Conv1d(1, 8, 64, stride=32), nn.Dropout(0.2), nn.Linear(8, 2)

    def forward(self, waveforms, valid_samples):
        x = torch.relu(self.conv(waveforms.unsqueeze(1)))
        frames = ((valid_samples - 64) // 32 + 1).clamp(min=1)
        mask = torch.arange(x.shape[-1], device=x.device)[None, :] < frames[:, None]
        return self.head(self.dropout((x * mask[:, None, :]).sum(-1) / frames[:, None]))


class SpectralBiGRU(nn.Module):
    def __init__(self, dropout=0.2):
        super().__init__()
        self.gru = BiGRU(64, 64, dropout)
        self.register_buffer("window", torch.hann_window(400))

    def forward(self, waveforms, valid_samples):
        power = torch.stft(waveforms, 512, 160, 400, window=self.window, center=False, return_complex=True).abs().pow(2)
        feats = torch.log(power[:, :256].reshape(power.shape[0], 64, 4, -1).mean(2) + 1e-6).transpose(1, 2)
        lengths = ((valid_samples - 512) // 160 + 1).clamp(1, feats.shape[1])
        return self.gru((feats + 5.0) / 5.0, lengths)


def build(name, factory, cfg, subdir):
    torch.manual_seed(0)
    train, ev = make_data()
    base = dict(seed=42, max_epochs=4, batch_size=8, lr=3e-3, patience=10, owner="r1-m1-colab", model_name=name, protocol="B")
    base.update(cfg)
    run = WORK / subdir / "run"
    return Trainer(factory(), TrainConfig(**base), train, ev, run, ledger_path=WORK / subdir / "ledger.csv",
                   provenance={"case": name}), run


def run_case(name, factory, **cfg):
    out = {"case": name, "status": "passed", "config": cfg}
    t0 = time.time()
    try:
        trainer, run = build(name, factory, cfg, name)
        result = trainer.fit()
        history = [json.loads(l) for l in (run / "history.jsonl").read_text().splitlines()]
        out.update(val_macro_f1_by_epoch=[round(h["val"]["macro_f1"], 3) for h in history], lr_by_epoch=[h["lr"] for h in history],
                   skipped_steps=history[-1]["skipped_steps"],
                   best_macro_f1=result["best_macro_f1"], epochs_run=result["epochs_run"], device=str(trainer.device),
                   peak_vram_mb=round(torch.cuda.max_memory_allocated() / 2**20, 1) if torch.cuda.is_available() else None)
        if result["best_macro_f1"] < 0.9:
            out.update(status="failed", error=f"model did not learn the separable fixture (best macro F1 {result['best_macro_f1']:.3f})")
    except Exception as exc:  # recorded, never raised: one report shows every problem
        out.update(status="failed", error=f"{type(exc).__name__}: {str(exc)[:300]}")
    out["seconds"] = round(time.time() - t0, 1)
    return out


def resume_case(name, factory, **cfg):
    out = {"case": name, "status": "passed", "config": cfg}
    try:
        full, full_dir = build(name, factory, {**cfg, "max_epochs": 3}, name + "_full")
        full.fit()
        part, part_dir = build(name, factory, {**cfg, "max_epochs": 3}, name + "_part")
        original = part._train_epoch

        def interrupt(epoch):
            if epoch == 3:
                raise KeyboardInterrupt
            return original(epoch)
        part._train_epoch = interrupt
        try:
            part.fit()
        except KeyboardInterrupt:
            pass
        resumed, _ = build(name, factory, {**cfg, "max_epochs": 3}, name + "_part")
        resumed.fit(resume=True)
        read = lambda d: [json.loads(l) for l in (d / "history.jsonl").read_text().splitlines()]
        a, b = read(full_dir), read(part_dir)
        diffs = [abs(x["train_loss"] - y["train_loss"]) for x, y in zip(a, b)] + [abs(x["val"]["loss"] - y["val"]["loss"]) for x, y in zip(a, b)]
        out.update(epochs_compared=len(a), max_abs_loss_difference=max(diffs))
        if len(a) != len(b) or max(diffs) > 1e-5:
            out.update(status="failed", error=f"resumed run diverged from the uninterrupted run (max diff {max(diffs):.2e})")
    except Exception as exc:
        out.update(status="failed", error=f"{type(exc).__name__}: {str(exc)[:300]}")
    return out

In [ ]:
CUDA = torch.cuda.is_available()
WINDOWS = os.name == "nt"
results = []

def go(entry):
    results.append(entry)
    print(f"{entry['case']:<28} {entry['status']:<8} {entry.get('error', '')}")

# required: these must pass for R1-M1 to be considered verified on this runtime
go(run_case("tiny_fp32_deterministic", TinyDetector, deterministic=True))
go(resume_case("tiny_resume_exact", TinyDetector, deterministic=True))
# batch_size=4 x accumulation 2 keeps 16 optimizer steps (32 clips); with only 8 steps this tiny model cannot learn at all
go(run_case("tiny_plateau_accum2", TinyDetector, scheduler="plateau", accumulation_steps=2, batch_size=4, grad_clip=1.0))
go(run_case("tiny_class_weights", TinyDetector, class_strategy="weights"))
if WINDOWS:
    results.append({"case": "tiny_dataloader_workers", "status": "skipped", "error": "Windows spawn"})
else:
    go(run_case("tiny_dataloader_workers", TinyDetector, num_workers=2))
if CUDA:
    # cosine decays to ~0, so it needs more steps than the constant-lr cases (8 epochs = 32 steps)
    go(run_case("tiny_fp16_amp_cosine", TinyDetector, precision="fp16", scheduler="cosine", warmup_steps=2, accumulation_steps=2, batch_size=4, max_epochs=8))
    go(run_case("tiny_bf16", TinyDetector, precision="bf16")) if torch.cuda.is_bf16_supported() else None
else:
    results.append({"case": "tiny_fp16_amp_cosine", "status": "skipped", "error": "no GPU"})

# BiGRU: strict deterministic algorithms can be refused by CUDA RNN kernels; the fallback must work
go(run_case("bigru_deterministic", SpectralBiGRU, deterministic=True, lr=3e-3))
go(run_case("bigru_nondeterministic", SpectralBiGRU, deterministic=False, lr=3e-3))
# Dropout is off here on purpose: cuDNN keeps a private dropout state for nn.GRU that no checkpoint can capture (see the
# limitation noted in the Trainer docstring), so exact resume can only be asserted for the BiGRU without dropout.
go(resume_case("bigru_resume", lambda: SpectralBiGRU(dropout=0.0), deterministic=True, lr=3e-3))

tiny_fp32_deterministic      passed   
tiny_resume_exact            passed   
tiny_plateau_accum2          passed   
tiny_class_weights           passed   
tiny_dataloader_workers      passed   
tiny_fp16_amp_cosine         passed   
tiny_bf16                    passed   
bigru_deterministic          passed   
bigru_nondeterministic       passed   
bigru_resume                 passed   


## 3. Verdict and evidence

In [ ]:
OPTIONAL = {"bigru_deterministic"}          # informational: tells you which setting to use for the BiGRU
required = [r for r in results if r["case"] not in OPTIONAL and r["status"] != "skipped"]
failed = [r["case"] for r in required if r["status"] != "passed"]
bigru_det = next((r for r in results if r["case"] == "bigru_deterministic"), {})

verdict = {
    "task": "R1-M1", "unit_tests_passed": UNIT_TESTS_PASSED, "required_cases_failed": failed,
    "gpu_cases_run": CUDA,
    "informational": {r["case"]: {"status": r["status"], "max_abs_loss_difference": r.get("max_abs_loss_difference"), "error": r.get("error")}
                      for r in results if r["case"] in OPTIONAL}, "verified": UNIT_TESTS_PASSED and not failed and CUDA,
    "recommendation": ("BiGRU may use deterministic=True" if bigru_det.get("status") == "passed"
                       else "set deterministic=False for BiGRU training on CUDA: " + str(bigru_det.get("error", "not run"))[:200]),
}
evidence = {"schema_version": 1, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "environment": ENV,
            "trainer_sha256": TRAINER_SHA256, "verdict": verdict, "cases": results}

out_dir = ROOT / "reports" / "r1"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"r1_m1_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")

print(json.dumps(verdict, indent=2))
print("\nevidence written to:", path)
if not verdict["verified"]:
    print("\nNOT VERIFIED:",
          "no GPU runtime;" if not CUDA else "", "unit tests failed;" if not UNIT_TESTS_PASSED else "",
          f"failed cases: {failed}" if failed else "")

{
  "task": "R1-M1",
  "unit_tests_passed": true,
  "required_cases_failed": [],
  "gpu_cases_run": true,
  "informational": {
    "bigru_deterministic": {
      "status": "passed",
      "max_abs_loss_difference": null,
      "error": null
    }
  },
  "verified": true,
  "recommendation": "BiGRU may use deterministic=True"
}

evidence written to: /content/drive/MyDrive/PAI&AIES Project/reports/r1/r1_m1_colab_20261103T182916Z.json
